# Conditional Cooperation — end-to-end walkthrough

This notebook loads the Herrmann/Thöni/Gächter (2008) Public Goods Game data,
fits the per-subject Conditional Cooperation model with PyMC, and produces the
national-level scatter plots that appear in the project README.

Run it from the repo root (so the data path is relative) with:

```bash
uv run --extra dev jupyter lab notebooks/01_walkthrough.ipynb
```

## 1. Load and inspect the data

In [ ]:
from pathlib import Path

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pymc as pm

from conditional_cooperation.data import load_public_goods_data
from conditional_cooperation.model import build_individual_model

DATA = Path('../data/public_good/HerrmannThoeniGaechterDATA.csv')
ds = load_public_goods_data(DATA, covariate='worry')
print(f'{ds.ngroups} groups across {ds.nnations} nations')
print('Nations:', ds.nation_names)
print('Worry index per nation:', ds.covariate.round(3))

## 2. Fit the per-subject CC model

We use the no-punishment condition (the one analysed in the original 2008 paper)
and the NUTS-friendly default priors (`alpha ~ Gamma(2, 0.1)`, `rho/omega ~ Beta(2, 2)`).

A 500-tune / 500-draw run takes ~1 minute on a modern laptop. For a faster fit,
install the optional `fast` extra (`uv sync --extra fast`) and pass
`nuts_sampler='numpyro'` to `pm.sample()` below.

In [ ]:
c = ds.c[:, :, :, 0]   # no-punishment
Ga = ds.Ga[:, :, :, 0]

with build_individual_model(c, Ga):
    idata = pm.sample(
        draws=500,
        tune=500,
        chains=2,
        target_accept=0.9,
        random_seed=1983,
        # nuts_sampler='numpyro',  # ← uncomment for the JAX backend
    )

## 3. Convergence diagnostics

In [ ]:
summary = az.summary(idata, var_names=['alpha', 'rho', 'omega'], kind='diagnostics')
print(f"max R-hat:    {summary['r_hat'].max():.3f}")
print(f"min ESS bulk: {summary['ess_bulk'].min():.0f}")
print(f"divergences:  {int(idata.sample_stats['diverging'].sum().item())}")

## 4. Per-nation aggregation

Posterior mean of each parameter, averaged across subjects in each nation.

In [ ]:
post_means = {
    name: idata.posterior[name].mean(('chain', 'draw')).values
    for name in ('alpha', 'rho', 'omega')
}

nations = np.unique(ds.nation)
per_nation = {
    name: np.array([mat[:, ds.nation == n].mean() for n in nations])
    for name, mat in post_means.items()
}

for name, vals in per_nation.items():
    r = np.corrcoef(ds.covariate, vals)[0, 1]
    print(f'{name:>5}: r(worry, {name}) = {r:+.2f}')

## 5. Scatter plot — covariate vs each CC parameter

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
for ax, name, ylabel, color in zip(
    axes,
    ('alpha', 'rho', 'omega'),
    ('Initial belief (α)', 'Matching preference (ρ)', 'Belief update (ω)'),
    ('#1f77b4', '#2ca02c', '#d62728'),
    strict=True,
):
    x, y = ds.covariate, per_nation[name]
    ax.scatter(x, y, s=80, color=color, edgecolor='black', alpha=0.85)
    for xi, yi, lbl in zip(x, y, ds.nation_names, strict=True):
        ax.annotate(lbl, (xi, yi), xytext=(4, 4), textcoords='offset points', fontsize=8)
    slope, intercept = np.polyfit(x, y, 1)
    xs = np.linspace(x.min(), x.max(), 100)
    ax.plot(xs, slope * xs + intercept, '--', color='gray', alpha=0.6)
    r = np.corrcoef(x, y)[0, 1]
    ax.set_xlabel('National worry index')
    ax.set_ylabel(ylabel)
    ax.set_title(f'r = {r:+.2f}')
    ax.grid(alpha=0.3)
plt.show()

## 6. Posterior trace for a few parameters

Sanity check that chains have mixed.

In [ ]:
az.plot_trace(idata, var_names=['alpha', 'rho', 'omega'], compact=True)
plt.tight_layout()
plt.show()

## Next steps

- Run the full hierarchical model (`cc-fit-worry`) to get credible intervals on the
  national-level *slopes* (and not just point estimates of nation means).
- Compare the no-punishment and punishment conditions side by side.
- Reparameterise the hierarchical model in non-centred form to fix the funnel
  geometry that currently produces divergences in NUTS.